In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder \
            .master('local[*]') \
            .appName('sql') \
            .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 07:07:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df_green = spark.read \
        .option("recursiveFileLookup", "true") \
        .option("pathGlobFilter", "*.snappy.parquet") \
        .parquet('data/pq/green')
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [5]:
df_green.createOrReplaceTempView('green')

In [8]:
df_green_revenue = spark.sql("""
SELECT
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS revenue_zone,
    SUM(total_amount) AS revenue_monthly_total_amount,
    COUNT(1) AS number_records
FROM 
    green
WHERE 
    lpep_pickup_datetime >= '2020-01-01'
GROUP BY
    1,2
ORDER BY
    1,2
""")

In [9]:
df_green_revenue.show()

+-------------------+------------+----------------------------+--------------+
|               hour|revenue_zone|revenue_monthly_total_amount|number_records|
+-------------------+------------+----------------------------+--------------+
|2020-01-01 00:00:00|           7|           769.7299999999998|            45|
|2020-01-01 00:00:00|          17|          195.03000000000003|             9|
|2020-01-01 00:00:00|          18|                         7.8|             1|
|2020-01-01 00:00:00|          22|                        15.8|             1|
|2020-01-01 00:00:00|          24|                        87.6|             3|
|2020-01-01 00:00:00|          25|           531.0000000000001|            26|
|2020-01-01 00:00:00|          29|                        61.3|             1|
|2020-01-01 00:00:00|          32|           68.94999999999999|             2|
|2020-01-01 00:00:00|          33|                      317.27|            11|
|2020-01-01 00:00:00|          35|                  

In [10]:
df_green_revenue.write.parquet('data/report/revenue/green')

26/03/14 07:14:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/14 07:14:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/14 07:14:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/14 07:14:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/14 07:14:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/14 07:14:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/14 07:14:10 WARN MemoryManager: Total allocation exceeds 95.0